In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""


In [3]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = "deepset/roberta-base-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)


QA_input = {
    "question": "What are the payment options available?",
    "context": "In our e-commerce, you can make payments using credit card, debit card, Apple Pay, or through bank transfers."
}

inputs = tokenizer(
    QA_input["question"],
    QA_input["context"],
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = model(**inputs)

start_index = torch.argmax(outputs.start_logits)
end_index = torch.argmax(outputs.end_logits)

answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

answer

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10059.50it/s]


' credit card, debit card, Apple Pay, or through bank transfers'

In [ ]:
def get_response(question, context):
    _inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
    with torch.no_grad():
        _outputs = model(**inputs)

        _start_index = torch.argmax(_outputs.start_logits)
        _end_index = torch.argmax(_outputs.end_logits)

    _answer_tokens = _inputs["input_ids"][0][_start_index:_end_index + 1]
    return tokenizer.decode(_answer_tokens, skip_special_tokens=True)

In [ ]:
get_response('What are the payment options available?', 'In our e-commerce, you can make payments using credit card, debit card, Apple Pay, or through bank transfers.')

In [ ]:
contexts = {
    "How do I create an account?": "You can create an account by clicking on the 'Sign Up' button on our homepage. You will need to provide your email address, create a password, and fill in some basic information about yourself. An account will help you track your orders, manage your personal settings, and speed up future transactions.",
    "Which payment methods do you accept?": "We accept a wide range of payment methods including Visa, MasterCard, American Express, Discover, PayPal, and Apple Pay. You can also pay using store credit or gift cards issued by our company.",
    "How can I track my order?": "Once your order has shipped, you will receive an email with a tracking number. You can use this number on our website's tracking page to see the current status of your delivery.",
    "Do you offer international shipping?": "Yes, we offer international shipping to most countries. Shipping costs and delivery times vary depending on the destination. All applicable customs fees, taxes, and duties are the responsibility of the customer and are calculated at checkout.",
    "How long does delivery take?": "For standard shipping, deliveries typically take between 3 to 5 business days. For expedited shipping, expect your order to arrive within 1 to 2 business days. Delivery times may vary based on your location and the time of the year.",
    "What is your return policy?": "Our return policy allows you to return products within 30 days of receiving them. Items must be in their original condition and packaging. Some items, such as perishable goods, are not eligible for return.",
    "Can I change or cancel my order after it's been placed?": "You can change or cancel your order within 24 hours of placing it without any additional charge.To make changes or cancel your order, please contact our customer service immediately.",
    "What should I do if I receive a damaged item?": "If you receive a damaged item, please contact our customer service within 48 hours of delivery to report the damage. You will need to provide your order number, the description of the damage, and photographic evidence. We will arrange for a replacement or refund as appropriate.",
    "How do I reset my password?": "If you've forgotten your password, go to the login page and click on 'Forgot Password'. Enter your email address and we will send you a link to reset your password. For security purposes, this link will expire within 24 hours."
}

In [ ]:
def answer_faq(question):
    context = contexts[question]
    _result = get_response(question, context)
    return _result

In [ ]:
answer_faq('How do I create an account?')

In [ ]:
answer_faq('Which payment methods do you accept?')

In [ ]:
answer_faq('Do you offer international shipping?')

In [ ]:
import gradio as gr

app = gr.Interface(
    fn=answer_faq,
    inputs=gr.Dropdown(choices=list(contexts.keys()), label="Select your question"),
    outputs='text',
    title='E-commerce FAQ',
    description='Select a question to get an answer from our FAQ.')
app.launch(share=True)